# Lab 3.1: Agentes com o Agent Service do Azure AI Foundry (20 min)

## Objetivos
- Criar um **agente persistente** no Azure AI Foundry (visível no portal!)
- Entender o ciclo: **Agent → Thread → Message → Run**
- Usar **function calling** com agentes
- Usar o **Code Interpreter**
- Gerir o ciclo de vida do agente (criar, conversar, eliminar)

## Conceitos

### Agente vs Chamada ao Modelo
No Lab 3 usámos a **Responses API** — chamadas diretas ao modelo. Aqui vamos criar **agentes persistentes** com o **Agent Service**, visíveis no portal do Foundry:

| Responses API (Lab 3) | Agent Service (Lab 3.1) |
|---|---|
| Chamada stateless | Agente persistente no Foundry |
| Sem memória entre chamadas | Threads mantêm contexto |
| Tu geres o loop de tools | O Run processa tools automaticamente |
| Ideal para uso simples | Ideal para assistentes complexos |

### Ciclo de Vida
```
1. Criar Agente (com instruções e tools) → aparece no portal Foundry
2. Criar Thread (conversa)
3. Adicionar Mensagem ao Thread
4. Criar Run (execução)
5. Processar Resposta (pode incluir tool calls)
6. Repetir 3-5 para continuar a conversa
7. Eliminar Agente quando já não for necessário
```

### SDK
```python
from azure.ai.agents import AgentsClient
from azure.identity import DefaultAzureCredential

agents_client = AgentsClient(
    endpoint=AGENT_ENDPOINT,
    credential=DefaultAzureCredential(),
)
```

In [1]:
# Setup
from dotenv import load_dotenv
import os
import json

load_dotenv("../.env")

from azure.ai.agents import AgentsClient
from azure.ai.agents.models import FunctionTool, ToolSet, CodeInterpreterTool
from azure.identity import DefaultAzureCredential

agent_endpoint = os.getenv("AZURE_AI_AGENT_ENDPOINT")
model = os.getenv("MODEL_DEPLOYMENT", "gpt-4o")

agents_client = AgentsClient(
    endpoint=agent_endpoint,
    credential=DefaultAzureCredential(),
)

print(f"✅ Agent Service conectado: {agent_endpoint}")

✅ Agent Service conectado: https://foundry-mod2.services.ai.azure.com/api/projects/proj-tutor


## 🖥️ Exercício 3.1.1: Agente Simples

Vamos criar um agente persistente com instruções e conversar com ele através de **threads**.

In [2]:
# Exercício 3.1.1: Criar um agente persistente (visível no portal Foundry!)
agent = agents_client.create_agent(
    model=model,
    name="assistente-azure",
    instructions="És um especialista em Azure. Responde de forma concisa em português de Portugal. Usa exemplos práticos.",
)
print(f"✅ Agente criado: {agent.id}")
print(f"   Nome: {agent.name}")
print(f"   Modelo: {agent.model}")
print(f"\n💡 Vai ao portal do Foundry e verifica que o agente aparece lá!")

✅ Agente criado: asst_Y97LciJGo3jJ91d87QkaLjoi
   Nome: assistente-azure
   Modelo: gpt-4o

💡 Vai ao portal do Foundry e verifica que o agente aparece lá!


In [3]:
# Criar thread (conversa) e enviar primeira mensagem
thread = agents_client.threads.create()
print(f"✅ Thread criada: {thread.id}")

agents_client.messages.create(
    thread_id=thread.id,
    role="user",
    content="Explica a diferença entre Azure Functions e Container Apps em 3 linhas.",
)

# Executar o agente no thread
run = agents_client.runs.create_and_process(
    thread_id=thread.id,
    agent_id=agent.id,
)
print(f"   Run status: {run.status}")

# Obter resposta (mensagens vêm da mais recente para a mais antiga)
for msg in reversed(list(agents_client.messages.list(thread_id=thread.id))):
    role = "👤" if msg.role == "user" else "🤖"
    for item in msg.content:
        if hasattr(item, "text"):
            print(f"{role} {item.text.value}")

✅ Thread criada: thread_09plr6FJejLCMGQ0kMkK0YLF
   Run status: RunStatus.COMPLETED
👤 Explica a diferença entre Azure Functions e Container Apps em 3 linhas.
🤖 O Azure Functions é um serviço serverless para executar código em resposta a eventos com escalabilidade automática, ideal para tarefas pequenas e orientadas por eventos, como processamento de filas. O Azure Container Apps permite executar aplicações em contentores, suportando microserviços complexos, HTTP e workloads de longa duração com controlo avançado de escalabilidade. Resumindo, Functions foca-se em simplicidade e eventos, enquanto Container Apps é mais flexível para aplicações em contentores.


In [4]:
# Continuar a conversa no MESMO thread (o agente mantém contexto!)
agents_client.messages.create(
    thread_id=thread.id,
    role="user",
    content="E qual dos dois recomendas para um webhook simples?",
)

run2 = agents_client.runs.create_and_process(
    thread_id=thread.id,
    agent_id=agent.id,
)
print(f"Run status: {run2.status}\n")

# Mostrar toda a conversa
for msg in reversed(list(agents_client.messages.list(thread_id=thread.id))):
    role = "👤" if msg.role == "user" else "🤖"
    for item in msg.content:
        if hasattr(item, "text"):
            print(f"{role} {item.text.value}\n")

Run status: RunStatus.COMPLETED

👤 Explica a diferença entre Azure Functions e Container Apps em 3 linhas.

🤖 O Azure Functions é um serviço serverless para executar código em resposta a eventos com escalabilidade automática, ideal para tarefas pequenas e orientadas por eventos, como processamento de filas. O Azure Container Apps permite executar aplicações em contentores, suportando microserviços complexos, HTTP e workloads de longa duração com controlo avançado de escalabilidade. Resumindo, Functions foca-se em simplicidade e eventos, enquanto Container Apps é mais flexível para aplicações em contentores.

👤 E qual dos dois recomendas para um webhook simples?

🤖 Recomendo o Azure Functions para um webhook simples, pois é mais fácil de configurar, orientado por eventos e tem custos baixos devido ao modelo pay-per-use. Por exemplo, podes criar uma Function para processar chamadas HTTP num minuto e pagar apenas pelas execuções feitas.



## 🖥️ Exercício 3.1.2: Agente com Function Calling

O verdadeiro poder dos agentes é usar **ferramentas**. Vamos criar um agente que pode consultar informações sobre preços e regiões Azure.

In [5]:
# Exercício 3.1.2: Definir funções e criar agente com tools

# Funções que o agente pode chamar
def obter_preco_servico(servico: str) -> str:
    """Retorna o preço estimado de um serviço Azure."""
    precos = {
        "app service basic": "€50/mês",
        "container apps": "€0.000012/vCPU-s",
        "functions consumption": "€0.000016/GB-s",
        "aks": "Grátis (control plane) + custo dos nodes",
        "cosmos db": "A partir de €25/mês (serverless)",
    }
    return json.dumps({"servico": servico, "preco": precos.get(servico.lower(), "Preço não disponível. Consulta azure.com/pricing")})

def obter_regiao_recomendada(caso_uso: str) -> str:
    """Recomenda a melhor região Azure para um caso de uso."""
    regioes = {
        "europa": "West Europe (Holanda) ou North Europe (Irlanda)",
        "portugal": "West Europe (mais próximo de Portugal)",
        "global": "Usa Traffic Manager com múltiplas regiões",
        "ai": "East US ou Sweden Central (melhor disponibilidade de modelos)",
    }
    return json.dumps({"caso_uso": caso_uso, "recomendacao": regioes.get(caso_uso.lower(), regioes["europa"])})

# Registar as funções como tools (o SDK extrai automaticamente os schemas!)
functions = FunctionTool(functions={obter_preco_servico, obter_regiao_recomendada})
toolset = ToolSet()
toolset.add(functions)

# Criar agente com toolset
agent_tools = agents_client.create_agent(
    model=model,
    name="consultor-azure",
    instructions="És um consultor Azure. Usa as ferramentas disponíveis para dar informações precisas sobre preços e regiões. Responde em português.",
    toolset=toolset,
)
print(f"✅ Agente com tools criado: {agent_tools.id}")

# Ativar processamento automático de function calls
agents_client.enable_auto_function_calls(toolset)

✅ Agente com tools criado: asst_I6R7kOJywcVVua6z9AXSIK0A


In [6]:
# Testar o agente com function calling
thread2 = agents_client.threads.create()
agents_client.messages.create(
    thread_id=thread2.id,
    role="user",
    content="Quero criar uma app com Container Apps. Quanto custa e qual a melhor região para Portugal?",
)

# O SDK processa automaticamente os tool calls graças ao enable_auto_function_calls!
run3 = agents_client.runs.create_and_process(
    thread_id=thread2.id,
    agent_id=agent_tools.id,
)
print(f"Run status: {run3.status}\n")

# Mostrar resultado
for msg in reversed(list(agents_client.messages.list(thread_id=thread2.id))):
    role = "👤" if msg.role == "user" else "🤖"
    for item in msg.content:
        if hasattr(item, "text"):
            print(f"{role} {item.text.value}\n")

Run status: RunStatus.COMPLETED

👤 Quero criar uma app com Container Apps. Quanto custa e qual a melhor região para Portugal?

🤖 O custo estimado para o serviço de **Container Apps** no Azure é de **€0.000012 por vCPU segundo**.

Para uma aplicação com proximidade a **Portugal**, as melhores regiões recomendadas são: **West Europe (Holanda)** ou **North Europe (Irlanda)**.



## 🖥️ Exercício 3.1.3: Code Interpreter

O **Code Interpreter** permite ao agente escrever e executar código Python autonomamente.

In [7]:
# Exercício 3.1.3: Agente com Code Interpreter
agent_code = agents_client.create_agent(
    model=model,
    name="analista-dados",
    instructions="És um analista de dados. Usa o code interpreter para fazer cálculos e criar visualizações. Responde em português.",
    tools=CodeInterpreterTool().definitions,
)
print(f"✅ Agente com code interpreter: {agent_code.id}")

thread3 = agents_client.threads.create()
agents_client.messages.create(
    thread_id=thread3.id,
    role="user",
    content="Calcula a sequência de Fibonacci até ao 10º número e mostra o resultado numa tabela formatada.",
)

run4 = agents_client.runs.create_and_process(
    thread_id=thread3.id,
    agent_id=agent_code.id,
)
print(f"   Run status: {run4.status}\n")

for msg in reversed(list(agents_client.messages.list(thread_id=thread3.id))):
    if msg.role == "assistant":
        for item in msg.content:
            if hasattr(item, "text"):
                print(f"🤖 {item.text.value}")

✅ Agente com code interpreter: asst_6WwDxgdIz8Mv0Uq8Im8OIIkI
   Run status: RunStatus.COMPLETED

🤖 Aqui está a sequência de Fibonacci até ao 10º número, apresentada numa tabela:

| Posição | Número de Fibonacci |
|---------|----------------------|
| 1       | 0                    |
| 2       | 1                    |
| 3       | 1                    |
| 4       | 2                    |
| 5       | 3                    |
| 6       | 5                    |
| 7       | 8                    |
| 8       | 13                   |
| 9       | 21                   |
| 10      | 34                   |


## 🧹 Limpeza

Os agentes são **persistentes** — é importante eliminá-los quando já não são necessários.

In [8]:
# Limpeza - eliminar todos os agentes criados
for a in [agent, agent_tools, agent_code]:
    agents_client.delete_agent(a.id)
    print(f"🗑️ Agente {a.name} eliminado ({a.id})")

print("\n✅ Limpeza concluída! Verifica no portal do Foundry que os agentes desapareceram.")

🗑️ Agente assistente-azure eliminado (asst_Y97LciJGo3jJ91d87QkaLjoi)
🗑️ Agente consultor-azure eliminado (asst_I6R7kOJywcVVua6z9AXSIK0A)
🗑️ Agente analista-dados eliminado (asst_6WwDxgdIz8Mv0Uq8Im8OIIkI)

✅ Limpeza concluída! Verifica no portal do Foundry que os agentes desapareceram.


## ✅ Resumo

Neste lab aprendeste:
- Criar **agentes persistentes** com o Agent Service do Foundry (visíveis no portal!)
- O ciclo **Agent → Thread → Message → Run**
- Manter **contexto** entre mensagens usando threads
- Usar **function calling** com processamento automático via `ToolSet`
- Usar o **Code Interpreter** para computação autónoma
- Gerir o ciclo de vida (criar e eliminar agentes)

### Lab 3 vs Lab 3.1
| Lab 3 (Responses API) | Lab 3.1 (Agent Service) |
|---|---|
| Chamadas diretas ao modelo | Agentes persistentes no portal Foundry |
| Tu geres o contexto | Threads mantêm o histórico |
| Tu processas tool calls | `enable_auto_function_calls` faz tudo |
| Mais simples e leve | Mais poderoso para assistentes complexos |

**Próximo:** [Lab 4 - Knowledge e RAG](lab04-knowledge-rag.ipynb)